In [0]:
df_hist = spark.table(
    "chilecompra.bronze.ordenes_compra_historical"
)

df_api = spark.table(
    "chilecompra.bronze.ordenes_compra_api"
)



In [0]:
from pyspark.sql.functions import col, expr

def date_col(column_name):
    return expr(
        f"""
        try_cast(
            CASE
                WHEN upper(trim(`{column_name}`)) IN ('', 'NA', 'N/A', 'NULL')
                THEN NULL
                ELSE trim(`{column_name}`)
            END
            AS DATE
        )
        """
    )

def decimal_col(column_name, precision=20, scale=6):
    return expr(
        f"""
        try_cast(
            replace(trim(`{column_name}`), ',', '.')
            AS DECIMAL({precision},{scale})
        )
        """
    )

In [0]:
df_hist_oc = (
    df_hist
    .select(
        col("codigo"),
        col("nombre"),
        col("descripcion_obervaciones").alias("descripcion_observaciones"),
        col("codigoestado").cast("int").alias("codigo_estado"),

        date_col("fechacreacion").alias("fecha_creacion"),
        date_col("fechaenvio").alias("fecha_envio"),
        date_col("fechaultimamodificacion").alias("fecha_ultima_modificacion"),
        date_col("fechaaceptacion").alias("fecha_aceptacion"),
        date_col("fechacancelacion").alias("fecha_cancelacion"),

        decimal_col("montototaloc").alias("monto_total"),
        decimal_col("montototaloc_pesoschilenos").alias("monto_total_clp"),
        decimal_col("totalnetooc").alias("total_neto"),
        decimal_col("impuestos").alias("impuestos"),
        decimal_col("descuentos").alias("descuentos"),
        decimal_col("cargos").alias("cargos"),

        decimal_col("porcentajeiva", 5, 2).alias("porcentaje_iva"),

        col("tipomonedaoc").alias("tipo_moneda"),
        col("codigoorganismopublico").alias("codigo_organismo_publico"),
        col("organismopublico").alias("organismo_publico"),
        col("codigoproveedor").alias("codigo_proveedor"),
        col("nombreproveedor").alias("nombre_proveedor"),
        col("codigolicitacion").alias("codigo_licitacion"),

        col("_ingestion_timestamp"),
        col("_source_year"),
        col("_source_month")
    )
)

In [0]:
df_hist_oc = (
    df_hist_oc
    .dropDuplicates(["codigo"])
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

w = (
    Window
    .partitionBy("codigo")
    .orderBy(
        col("_process_date").desc(),
        col("_source_created_at").desc(),
        col("_ingestion_timestamp").desc()
    )
)

df_api_latest = (
    df_api
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

In [0]:
estado_data = [
    (4, "Enviada a Proveedor"),
    (5, "En proceso"),
    (6, "Aceptada"),
    (7, "Solicitud de cancelación"),
    (9, "Cancelada"),
    (12, "Recepción Conforme"),
    (13, "Pendiente de Recepcionar"),
    (14, "Recepcionada Parcialmente"),
    (15, "Recepción Conforme Incompleta"),
]

df_estado_map = spark.createDataFrame(
    estado_data,
    ["codigo_estado", "estado"]
)

In [0]:
df_hist_oc = (
    df_hist_oc
    .join(
        df_estado_map,
        on="codigo_estado",
        how="left"
    )
)

df_api_latest = (
    df_api_latest
    .join(
        df_estado_map,
        on="codigo_estado",
        how="left"
    )
)

In [0]:
null_codigo_hist = df_hist_oc.filter(col("codigo").isNull()).count()
null_codigo_api = df_api_latest.filter(col("codigo").isNull()).count()

unmapped_hist = df_hist_oc.filter(col("estado").isNull()).count()
unmapped_api = df_api_latest.filter(col("estado").isNull()).count()

if null_codigo_hist > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo_hist} historical orders have codigo NULL"
    )

if null_codigo_api > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo_api} API orders have codigo NULL"
    )

if unmapped_hist > 0:
    raise ValueError(
        f"DQ FAILED: {unmapped_hist} historical rows have unmapped codigo_estado"
    )

if unmapped_api > 0:
    raise ValueError(
        f"DQ FAILED: {unmapped_api} API rows have unmapped codigo_estado"
    )

print("Pre-merge Silver DQ passed")

In [0]:
from pyspark.sql.functions import col, lit

df_hist_target = (
    df_hist_oc
    .select(
        "codigo",
        "nombre",
        "descripcion_observaciones",
        "codigo_estado",
        "estado",
        "fecha_creacion",
        "fecha_envio",
        "fecha_ultima_modificacion",
        "fecha_aceptacion",
        "fecha_cancelacion",
        "monto_total",
        "monto_total_clp",
        "total_neto",
        "impuestos",
        "descuentos",
        "cargos",
        "porcentaje_iva",
        "tipo_moneda",
        "codigo_organismo_publico",
        "organismo_publico",
        "codigo_proveedor",
        "nombre_proveedor",
        "codigo_licitacion",
        col("_source_year").alias("_historical_source_year"),
        col("_source_month").alias("_historical_source_month")
    )
    .withColumn("_last_api_process_date", lit(None).cast("date"))
    .withColumn("_last_api_source_created_at", lit(None).cast("timestamp"))
)

In [0]:
df_api_merge = (
    df_api_latest
    .select(
        col("codigo"),
        col("nombre"),

        lit(None).cast("string").alias("descripcion_observaciones"),

        col("codigo_estado"),
        col("estado"),

        lit(None).cast("date").alias("fecha_creacion"),
        lit(None).cast("date").alias("fecha_envio"),
        lit(None).cast("date").alias("fecha_ultima_modificacion"),
        lit(None).cast("date").alias("fecha_aceptacion"),
        lit(None).cast("date").alias("fecha_cancelacion"),

        lit(None).cast("decimal(20,6)").alias("monto_total"),
        lit(None).cast("decimal(20,6)").alias("monto_total_clp"),
        lit(None).cast("decimal(20,6)").alias("total_neto"),
        lit(None).cast("decimal(20,6)").alias("impuestos"),
        lit(None).cast("decimal(20,6)").alias("descuentos"),
        lit(None).cast("decimal(20,6)").alias("cargos"),
        lit(None).cast("decimal(5,2)").alias("porcentaje_iva"),

        lit(None).cast("string").alias("tipo_moneda"),
        lit(None).cast("string").alias("codigo_organismo_publico"),
        lit(None).cast("string").alias("organismo_publico"),
        lit(None).cast("string").alias("codigo_proveedor"),
        lit(None).cast("string").alias("nombre_proveedor"),
        lit(None).cast("string").alias("codigo_licitacion"),

        lit(None).cast("int").alias("_historical_source_year"),
        lit(None).cast("int").alias("_historical_source_month"),

        col("_process_date").alias("_last_api_process_date"),
        col("_source_created_at").alias("_last_api_source_created_at")
    )
)

In [0]:
from delta.tables import DeltaTable

target_table = "chilecompra.silver.ordenes_compra"

In [0]:
if not spark.catalog.tableExists(target_table):

    (
        df_hist_target.write
        .format("delta")
        .saveAsTable(target_table)
    )

else:

    delta_target = DeltaTable.forName(spark, target_table)

    (
        delta_target.alias("t")
        .merge(
            df_hist_target.alias("s"),
            "t.codigo = s.codigo"
        )
        .whenMatchedUpdate(
            set={
                # Si ya fue actualizado por API, no retrocedemos
                # nombre/estado a la versión histórica.
                "nombre":
                    "CASE WHEN t._last_api_process_date IS NULL "
                    "THEN s.nombre ELSE t.nombre END",

                "codigo_estado":
                    "CASE WHEN t._last_api_process_date IS NULL "
                    "THEN s.codigo_estado ELSE t.codigo_estado END",

                "estado":
                    "CASE WHEN t._last_api_process_date IS NULL "
                    "THEN s.estado ELSE t.estado END",

                "descripcion_observaciones": "s.descripcion_observaciones",
                "fecha_creacion": "s.fecha_creacion",
                "fecha_envio": "s.fecha_envio",
                "fecha_ultima_modificacion": "s.fecha_ultima_modificacion",
                "fecha_aceptacion": "s.fecha_aceptacion",
                "fecha_cancelacion": "s.fecha_cancelacion",

                "monto_total": "s.monto_total",
                "monto_total_clp": "s.monto_total_clp",
                "total_neto": "s.total_neto",
                "impuestos": "s.impuestos",
                "descuentos": "s.descuentos",
                "cargos": "s.cargos",
                "porcentaje_iva": "s.porcentaje_iva",

                "tipo_moneda": "s.tipo_moneda",
                "codigo_organismo_publico": "s.codigo_organismo_publico",
                "organismo_publico": "s.organismo_publico",
                "codigo_proveedor": "s.codigo_proveedor",
                "nombre_proveedor": "s.nombre_proveedor",
                "codigo_licitacion": "s.codigo_licitacion",

                "_historical_source_year": "s._historical_source_year",
                "_historical_source_month": "s._historical_source_month"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
delta_target = DeltaTable.forName(spark, target_table)

api_is_newer = """
    t._last_api_process_date IS NULL

    OR s._last_api_process_date > t._last_api_process_date

    OR (
        s._last_api_process_date = t._last_api_process_date
        AND (
            t._last_api_source_created_at IS NULL
            OR s._last_api_source_created_at > t._last_api_source_created_at
        )
    )
"""

In [0]:
(
    delta_target.alias("t")
    .merge(
        df_api_merge.alias("s"),
        "t.codigo = s.codigo"
    )
    .whenMatchedUpdate(
        condition=api_is_newer,
        set={
            "nombre": "s.nombre",
            "codigo_estado": "s.codigo_estado",
            "estado": "s.estado",
            "_last_api_process_date": "s._last_api_process_date",
            "_last_api_source_created_at": "s._last_api_source_created_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_silver = spark.table(target_table)
current_codes = (
    df_hist_oc
    .select("codigo")
    .union(df_api_latest.select("codigo"))
    .distinct()
)

missing_in_silver = (
    current_codes
    .join(
        df_silver.select("codigo"),
        on="codigo",
        how="left_anti"
    )
    .count()
)

silver_rows = df_silver.count()

silver_distinct = (
    df_silver
    .select("codigo")
    .distinct()
    .count()
)

silver_null_codigo = (
    df_silver
    .filter(col("codigo").isNull())
    .count()
)

if missing_in_silver > 0:
    raise ValueError(
        f"DQ FAILED: {missing_in_silver} current source orders "
        f"are missing from Silver"
    )

if silver_rows != silver_distinct:
    raise ValueError(
        f"DQ FAILED: rows={silver_rows}, "
        f"distinct codigo={silver_distinct}"
    )

if silver_null_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {silver_null_codigo} rows have codigo NULL"
    )

print(
    f"Silver DQ passed: "
    f"{silver_rows} orders, "
    f"all current source orders present"
)

In [0]:
df_api_vs_silver_invalid = (
    df_api_latest.alias("a")
    .join(
        df_silver.alias("s"),
        on="codigo",
        how="inner"
    )
    .filter(
        (col("a.codigo_estado") != col("s.codigo_estado")) |
        (col("a.estado") != col("s.estado"))
    )
    .filter(
        ~(
            (col("s._last_api_process_date") > col("a._process_date"))
            |
            (
                (col("s._last_api_process_date") == col("a._process_date"))
                &
                (
                    col("s._last_api_source_created_at")
                    > col("a._source_created_at")
                )
            )
        )
    )
)

invalid_mismatch_count = df_api_vs_silver_invalid.count()

if invalid_mismatch_count > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_mismatch_count} API states "
        f"should have been applied to Silver but were not"
    )

print("API vs Silver state DQ passed")